<a href="https://colab.research.google.com/github/exoxeph/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exoxeph/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

import duckdb
import pandas as pd
import numpy as np

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

api = HfApi()
me = api.whoami(token=hf_token)

print("Authenticated as:", me["name"])

Authenticated as: ExoCeph


In [9]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token,
)

content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=hf_token,
)

clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=hf_token,
)

print("Warehouse files ready.")

Warehouse files ready.


In [10]:
con = duckdb.connect()

print("DuckDB ready.")

DuckDB ready.


In [11]:
FEATURE_START = "2026-03-01"
DECISION_DATE = "2026-03-15"
LABEL_START = "2026-03-16"
LABEL_END = "2026-03-31"

print("Feature window:", FEATURE_START, "to", DECISION_DATE)
print("Outcome window:", LABEL_START, "to", LABEL_END)

Feature window: 2026-03-01 to 2026-03-15
Outcome window: 2026-03-16 to 2026-03-31


In [12]:
fact_schema = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
    """
).df()

content_schema = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{content_path}')
    """
).df()

clients_schema = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{clients_path}')
    """
).df()

print("Daily performance columns:", len(fact_schema))
print("Content columns:", len(content_schema))
print("Client columns:", len(clients_schema))

Daily performance columns: 31
Content columns: 26
Client columns: 9


In [13]:
MIN_GSC_DAYS = 8
DECLINE_RATIO = 0.80

print("Minimum GSC days per window:", MIN_GSC_DAYS)
print("Decline threshold:", f"{(1 - DECLINE_RATIO):.0%}")

Minimum GSC days per window: 8
Decline threshold: 20%


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I build one feature vector per eligible client-content item at the March 15 decision point. The numerical features are average daily GSC impressions, average daily GSC clicks, weighted average search position, CTR, and content age, all available by March 15. I also include content_type and main_intent as categorical metadata. Numerical missing values are median-filled, categorical missing values receive an explicit MISSING category, and categorical fields are one-hot encoded. Pseudonymous IDs and all future-window information are excluded from the feature matrix.

In [14]:
raw_model_df = con.sql(
    f"""
    WITH past AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions)::DOUBLE / COUNT(*)
                AS past_impressions_per_day,

            SUM(gsc_clicks)::DOUBLE / COUNT(*)
                AS past_clicks_per_day,

            SUM(gsc_sum_position)::DOUBLE
                / NULLIF(SUM(gsc_impressions), 0)
                AS past_avg_position,

            SUM(gsc_clicks)::DOUBLE
                / NULLIF(SUM(gsc_impressions), 0)
                AS past_ctr,

            COUNT(*) AS past_available_days

        FROM read_parquet('{march_path}')

        WHERE
            report_date BETWEEN
                DATE '{FEATURE_START}'
                AND DATE '{DECISION_DATE}'

            AND gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING
            COUNT(*) >= {MIN_GSC_DAYS}
            AND SUM(gsc_impressions) > 0
    ),

    future AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions)::DOUBLE / COUNT(*)
                AS future_impressions_per_day,

            COUNT(*) AS future_available_days

        FROM read_parquet('{march_path}')

        WHERE
            report_date BETWEEN
                DATE '{LABEL_START}'
                AND DATE '{LABEL_END}'

            AND gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING COUNT(*) >= {MIN_GSC_DAYS}
    )

    SELECT
        p.client_hash_id,
        p.content_hash_id,

        p.past_impressions_per_day,
        p.past_clicks_per_day,
        p.past_avg_position,
        p.past_ctr,

        DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '{DECISION_DATE}'
        ) AS content_age_days,

        c.content_type,
        c.main_intent,

        CASE
            WHEN f.future_impressions_per_day
                 <= {DECLINE_RATIO} * p.past_impressions_per_day
            THEN 1
            ELSE 0
        END AS is_declining

    FROM past AS p

    INNER JOIN future AS f
        ON p.client_hash_id = f.client_hash_id
       AND p.content_hash_id = f.content_hash_id

    INNER JOIN read_parquet('{content_path}') AS c
        ON p.client_hash_id = c.client_hash_id
       AND p.content_hash_id = c.content_hash_id

    WHERE
        c.content_created_date IS NOT NULL
        AND c.content_created_date <= DATE '{DECISION_DATE}'
    """
).df()

print("Rows:", len(raw_model_df))
display(raw_model_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 103710


,client_hash_id,content_hash_id,past_impressions_per_day,past_clicks_per_day,past_avg_position,past_ctr,content_age_days,content_type,main_intent,is_declining
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,3.800000,0.000000,3.964912,0.000000,31,keyword article,informational,1
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,331.600000,0.600000,8.088460,0.001809,31,keyword article,informational,0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,2.333333,0.000000,4.857143,0.000000,31,keyword article,commercial,0
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,26.533333,0.066667,5.298995,0.002513,31,keyword article,informational,1
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,2.846154,0.000000,17.837838,0.000000,31,keyword article,informational,1


In [18]:
print("Content type categories:")
print(raw_model_df["content_type"].value_counts(dropna=False))

print("\nMain intent categories:")
print(raw_model_df["main_intent"].value_counts(dropna=False))

print("\nUnique counts:")
print(
    raw_model_df[
        ["content_type", "main_intent"]
    ].nunique(dropna=False)
)

Content type categories:
content_type
keyword article       100510
comparison article      1962
feedly article          1238
Name: count, dtype: int64

Main intent categories:
main_intent
informational    60788
transactional    22475
commercial       18203
None              1979
navigational       265
Name: count, dtype: int64

Unique counts:
content_type    3
main_intent     5
dtype: int64


In [19]:
NUMERIC_FEATURES = [
    "past_impressions_per_day",
    "past_clicks_per_day",
    "past_avg_position",
    "past_ctr",
    "content_age_days",
]

CATEGORICAL_FEATURES = [
    "content_type",
    "main_intent",
]

TARGET = "is_declining"

X_raw = raw_model_df[
    NUMERIC_FEATURES + CATEGORICAL_FEATURES
].copy()

y = raw_model_df[TARGET].copy()

print("Raw X shape:", X_raw.shape)
print("y shape:", y.shape)

display(X_raw.head())

Raw X shape: (103710, 7)
y shape: (103710,)


,past_impressions_per_day,past_clicks_per_day,past_avg_position,past_ctr,content_age_days,content_type,main_intent
0,3.800000,0.000000,3.964912,0.000000,31,keyword article,informational
1,331.600000,0.600000,8.088460,0.001809,31,keyword article,informational
2,2.333333,0.000000,4.857143,0.000000,31,keyword article,commercial
3,26.533333,0.066667,5.298995,0.002513,31,keyword article,informational
4,2.846154,0.000000,17.837838,0.000000,31,keyword article,informational


In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="MISSING"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        ),
    ],
    verbose_feature_names_out=False,
)

X = preprocessor.fit_transform(X_raw)

feature_names = preprocessor.get_feature_names_out()

X = pd.DataFrame(
    X,
    columns=feature_names,
    index=X_raw.index,
)

print("Final feature-vector shape:", X.shape)
display(X.head())

Final feature-vector shape: (103710, 13)


,past_impressions_per_day,past_clicks_per_day,past_avg_position,past_ctr,content_age_days,content_type_comparison article,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_None
0,3.800000,0.000000,3.964912,0.000000,31.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,331.600000,0.600000,8.088460,0.001809,31.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,2.333333,0.000000,4.857143,0.000000,31.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
3,26.533333,0.066667,5.298995,0.002513,31.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,2.846154,0.000000,17.837838,0.000000,31.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [21]:
X_raw[CATEGORICAL_FEATURES] = (
    X_raw[CATEGORICAL_FEATURES]
    .replace({None: np.nan})
)

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

I use five numeric feature concepts and two categorical concepts. past_impressions_per_day measures average daily Google Search visibility during March 1–15. past_clicks_per_day measures average daily search clicks over the same past-only window. past_avg_position is an impression-weighted average search position, while past_ctr measures clicks relative to impressions. content_age_days measures the age of the content at the March 15 decision point. content_type and main_intent provide categorical content context and are one-hot encoded.

All seven feature concepts are available by the March 15 prediction moment; none uses March 16–31 data. Numerical missing values are median-filled. Missing categorical values receive an explicit MISSING category before one-hot encoding. Identifiers are not part of the feature vector.

In [22]:
feature_notes = pd.DataFrame(
    [
        {
            "feature": "past_impressions_per_day",
            "meaning": "Average daily GSC impressions from Mar 1-15",
            "missing_handling": "Median fill",
            "categorical": False,
            "available_when": "By Mar 15",
        },
        {
            "feature": "past_clicks_per_day",
            "meaning": "Average daily GSC clicks from Mar 1-15",
            "missing_handling": "Median fill",
            "categorical": False,
            "available_when": "By Mar 15",
        },
        {
            "feature": "past_avg_position",
            "meaning": "Impression-weighted average GSC position from Mar 1-15",
            "missing_handling": "Median fill",
            "categorical": False,
            "available_when": "By Mar 15",
        },
        {
            "feature": "past_ctr",
            "meaning": "Past GSC clicks divided by past impressions",
            "missing_handling": "Median fill",
            "categorical": False,
            "available_when": "By Mar 15",
        },
        {
            "feature": "content_age_days",
            "meaning": "Days from content creation to Mar 15",
            "missing_handling": "Median fill",
            "categorical": False,
            "available_when": "By Mar 15",
        },
        {
            "feature": "content_type",
            "meaning": "Content-format category",
            "missing_handling": "MISSING category",
            "categorical": True,
            "available_when": "By Mar 15",
        },
        {
            "feature": "main_intent",
            "meaning": "Search-intent category",
            "missing_handling": "MISSING category",
            "categorical": True,
            "available_when": "By Mar 15",
        },
    ]
)

feature_notes

,feature,meaning,missing_handling,categorical,available_when
0,past_impressions_per_day,Average daily GSC impressions from Mar 1-15,Median fill,False,By Mar 15
1,past_clicks_per_day,Average daily GSC clicks from Mar 1-15,Median fill,False,By Mar 15
2,past_avg_position,Impression-weighted average GSC position from ...,Median fill,False,By Mar 15
3,past_ctr,Past GSC clicks divided by past impressions,Median fill,False,By Mar 15
4,content_age_days,Days from content creation to Mar 15,Median fill,False,By Mar 15
5,content_type,Content-format category,MISSING category,True,By Mar 15
6,main_intent,Search-intent category,MISSING category,True,By Mar 15


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage hunt: I tested the final feature vector for target-derived fields, future-window information, product-rule shortcuts, and identifier-like fields. is_declining is kept only as the target and is not included in X. All honest performance features use March 1–15 observations, while March 16–31 is reserved for constructing the later decline proxy. I deliberately reconstructed future_impressions_per_day and future_visibility_ratio as examples of invalid features; both use the outcome window, and the ratio closely encodes the label rule itself. Pseudonymous client/content IDs and URL/query identifiers are also excluded from X. The automated checks confirm that none of these blocked fields are present in the final feature vector.

In [23]:
blocked_terms = [
    "future",
    "declin",
    "label",
    "target",
    "quick_win",
    "refresh_candidate",
    "health_score",
    "url",
    "hash_id",
]

leakage_hits = []

for feature in X.columns:
    matched = [
        term
        for term in blocked_terms
        if term.lower() in feature.lower()
    ]

    if matched:
        leakage_hits.append(
            {
                "feature": feature,
                "matched_terms": ", ".join(matched),
            }
        )

leakage_audit = pd.DataFrame(
    leakage_hits,
    columns=["feature", "matched_terms"]
)

print("Final feature columns:", X.shape[1])
print("Suspicious feature names found:", len(leakage_audit))

display(leakage_audit)

Final feature columns: 13
Suspicious feature names found: 0


,feature,matched_terms


In [24]:
forbidden_exact = {
    "is_declining",
    "client_hash_id",
    "content_hash_id",
    "future_impressions_per_day",
}

present_forbidden = forbidden_exact.intersection(X.columns)

print(
    "Forbidden exact fields found:",
    sorted(present_forbidden)
)

assert "is_declining" not in X.columns
assert "client_hash_id" not in X.columns
assert "content_hash_id" not in X.columns
assert not any(
    "future" in col.lower()
    for col in X.columns
)

print("Direct leakage checks passed.")

Forbidden exact fields found: []
Direct leakage checks passed.


In [25]:
future_check = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions)::DOUBLE / COUNT(*)
            AS future_impressions_per_day

    FROM read_parquet('{march_path}')

    WHERE
        report_date BETWEEN
            DATE '{LABEL_START}'
            AND DATE '{LABEL_END}'

        AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING COUNT(*) >= {MIN_GSC_DAYS}
    """
).df()

In [26]:
leak_demo = raw_model_df.merge(
    future_check,
    on=[
        "client_hash_id",
        "content_hash_id",
    ],
    how="left",
    validate="one_to_one",
)

leak_demo["future_visibility_ratio"] = (
    leak_demo["future_impressions_per_day"]
    / leak_demo["past_impressions_per_day"]
)

leak_demo[
    [
        "past_impressions_per_day",
        "future_impressions_per_day",
        "future_visibility_ratio",
        "is_declining",
    ]
].head()

,past_impressions_per_day,future_impressions_per_day,future_visibility_ratio,is_declining
0,3.800000,2.222222,0.584795,1
1,331.600000,367.187500,1.107321,0
2,2.333333,2.166667,0.928571,0
3,26.533333,19.187500,0.723147,1
4,2.846154,1.625000,0.570946,1


In [27]:
leaked_columns = [
    "future_impressions_per_day",
    "future_visibility_ratio",
]

for col in leaked_columns:
    print(
        col,
        "→ BLOCKED"
        if "future" in col.lower()
        else "review"
    )

future_impressions_per_day → BLOCKED
future_visibility_ratio → BLOCKED


In [28]:
privacy_terms = [
    "client_hash",
    "content_hash",
    "url",
    "keyword_hash",
]

privacy_hits = [
    col
    for col in X.columns
    if any(
        term in col.lower()
        for term in privacy_terms
    )
]

print(
    "Identifier/private-like fields in X:",
    privacy_hits
)

assert len(privacy_hits) == 0

print("Privacy feature check passed.")

Identifier/private-like fields in X: []
Privacy feature check passed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded fields: I excluded the target itself, all future-window variables, identifiers, and rule-derived shortcut fields from the model feature vector. is_declining is the label and therefore cannot appear in X. future_impressions_per_day and future_visibility_ratio use March 16–31 information, which is unavailable at the March 15 prediction moment and would create leakage. client_hash_id, content_hash_id, url_hash_id, and keyword_hash_id are identifiers or context fields rather than generalizable predictive signals. I also exclude product-rule flags because they may encode existing business logic instead of independent evidence. GA4-derived fields are left out of this first GSC-focused vector because historical GA4 availability is uneven and would require a separate availability treatment.

In [29]:
excluded_fields = pd.DataFrame(
    [
        {
            "field": "is_declining",
            "reason": "Target label; using it in X would directly reveal the answer.",
        },
        {
            "field": "future_impressions_per_day",
            "reason": "Uses Mar 16-31 data, which occurs after the Mar 15 prediction moment.",
        },
        {
            "field": "future_visibility_ratio",
            "reason": "Future-derived and closely encodes the rule used to create is_declining.",
        },
        {
            "field": "client_hash_id",
            "reason": "Identifier used for grouping/joining, not a generalizable model signal.",
        },
        {
            "field": "content_hash_id",
            "reason": "Identifier used for grouping/joining, not a generalizable model signal.",
        },
        {
            "field": "url_hash_id",
            "reason": "Identifier-like field that is unnecessary for prediction.",
        },
        {
            "field": "keyword_hash_id",
            "reason": "Identifier-like field that could encourage memorization.",
        },
        {
            "field": "product_rule_flags",
            "reason": "Could encode existing product/business logic and shortcut the learning task.",
        },
        {
            "field": "GA4-derived features",
            "reason": "Left out of this first GSC-focused vector because historical GA4 availability is uneven.",
        },
    ]
)

excluded_fields

,field,reason
0,is_declining,Target label; using it in X would directly rev...
1,future_impressions_per_day,"Uses Mar 16-31 data, which occurs after the Ma..."
2,future_visibility_ratio,Future-derived and closely encodes the rule us...
3,client_hash_id,"Identifier used for grouping/joining, not a ge..."
4,content_hash_id,"Identifier used for grouping/joining, not a ge..."
5,url_hash_id,Identifier-like field that is unnecessary for ...
6,keyword_hash_id,Identifier-like field that could encourage mem...
7,product_rule_flags,Could encode existing product/business logic a...
8,GA4-derived features,Left out of this first GSC-focused vector beca...


In [30]:
blocked_exact = {
    "is_declining",
    "client_hash_id",
    "content_hash_id",
    "future_impressions_per_day",
    "future_visibility_ratio",
    "url_hash_id",
    "keyword_hash_id",
}

found_blocked = [
    col for col in X.columns
    if col in blocked_exact
]

print("Blocked exact fields found in X:", found_blocked)

assert len(found_blocked) == 0

print("Exclusion check passed.")

Blocked exact fields found in X: []
Exclusion check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.